# XAUUSD – Exploratory Data Analysis (EDA)

Tento notebook slouží k prozkoumání datasetu před trénováním modelu.  
**Spusť buňky postupně od shora dolů.**

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_csv, create_target, drop_leaky_and_meta_columns

plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style='darkgrid')

# --- Nastav cestu ke svému CSV ---
DATA_PATH = '../data/xauusd.csv'

## 1. Načtení dat

In [ ]:
df_raw = load_csv(DATA_PATH)
df = create_target(df_raw)
df = drop_leaky_and_meta_columns(df)

print(f'Shape: {df.shape}')
df.head(3)

## 2. Základní statistiky

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

## 3. Balance targetu (Long vs Short)

In [ ]:
target_counts = df['target'].value_counts()
print(target_counts)
print(f'\nPoměr Long/Short: {target_counts[1]/target_counts[0]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_counts.plot.bar(ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Počet Long (1) vs Short (0)')
axes[0].set_xticklabels(['Short (0)', 'Long (1)'], rotation=0)

# Rolling balance po čase (kloubí se trh?)
df['target'].rolling(200).mean().plot(ax=axes[1])
axes[1].axhline(0.5, color='orange', linestyle='--', label='50 %')
axes[1].set_title('Rolling Long-ratio (200 svíček)')
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Chybějící hodnoty (NaN)

In [ ]:
nan_pct = df.isna().mean().sort_values(ascending=False)
nan_pct = nan_pct[nan_pct > 0]

if len(nan_pct) == 0:
    print('Žádné NaN hodnoty!')
else:
    print(f'Sloupce s NaN ({len(nan_pct)}):')
    nan_pct.plot.barh(figsize=(10, max(4, len(nan_pct) * 0.3)))
    plt.xlabel('Podíl NaN')
    plt.title('NaN ratio per feature')
    plt.tight_layout()
    plt.show()

## 5. Close cena + ATR v čase

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

df['close'].plot(ax=ax1, color='gold', linewidth=0.5)
ax1.set_title('XAUUSD Close Price')
ax1.set_ylabel('Price (USD)')

df['atr14'].plot(ax=ax2, color='steelblue', linewidth=0.5)
ax2.set_title('ATR14 (volatilita)')
ax2.set_ylabel('ATR')

plt.tight_layout()
plt.show()

## 6. Korelační matice (Top 30 features s targetem)

In [ ]:
corr_with_target = df.corrwith(df['target']).abs().sort_values(ascending=False)
top_features = corr_with_target.drop('target', errors='ignore').head(30).index.tolist()

corr_matrix = df[top_features + ['target']].corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0,
            annot=False, square=True, linewidths=0.3)
plt.title('Korelační matice – Top 30 features')
plt.tight_layout()
plt.show()

print('\nTop 10 features korelovaných s targetem:')
print(corr_with_target.drop('target', errors='ignore').head(10))

## 7. Distribuce klíčových features (Long vs Short)

In [ ]:
key_features = [
    'f30_dist_ema9_atr', 'f30_ofi_z200', 'f30_mtf_H1_ret',
    'eng_ret_atr_5', 'eng_vol_rank_200', 'f30_mtf_trend_alignment'
]
key_features = [f for f in key_features if f in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    df[df['target'] == 0][feat].plot.hist(
        ax=axes[i], bins=50, alpha=0.5, color='red', label='Short (0)', density=True)
    df[df['target'] == 1][feat].plot.hist(
        ax=axes[i], bins=50, alpha=0.5, color='green', label='Long (1)', density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribuce features: Long vs Short', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Sezónní vzory (hodina dne, den v týdnu)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

hourly = df.groupby(df.index.hour)['target'].mean()
hourly.plot.bar(ax=ax1, color='steelblue')
ax1.axhline(0.5, color='orange', linestyle='--')
ax1.set_title('Long ratio dle hodiny dne (UTC)')
ax1.set_xlabel('Hodina')
ax1.set_ylabel('Průměrný target (Long ratio)')

dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily = df.groupby(df.index.dayofweek)['target'].mean()
daily.index = [dow_names[i] for i in daily.index]
daily.plot.bar(ax=ax2, color='steelblue')
ax2.axhline(0.5, color='orange', linestyle='--')
ax2.set_title('Long ratio dle dne v týdnu')
ax2.set_xlabel('Den')

plt.tight_layout()
plt.show()